# DS6 — Network Theory (NeT) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS6 (Danish Adolescents, quasi-experimental, N=243 behavioral)
**Model family:** Network Theory (NeT)
**Date:** 2026-08-06
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds6_net_blueprint.md`

---

## Scope and purpose

Second DS6 CAS model, independent of StoP (73). Builds a directed app-transition network
per participant, separately for the baseline and intervention periods, and compares network
structure (hub-concentration, self-loop rate, transition entropy) across that boundary --
mirroring StoP's period-split design for direct cross-model comparability.

- **Goal:** Characterize DS6 participants' app-switching structure as a network; compare
  baseline vs intervention; bridge to StoP's volume finding (notebook 73).
- **Not goal:** Phenotype classification, personalization, cross-dataset harmonization.
- **Output:** Per-participant, per-period network metrics (HHI_net, rho_self, H_trans, H_pi),
  transition edge list, condition x period summary, and a StoP-NeT bridge comparison.

### FLAG decisions in effect (blueprint Part 2-4)

| Flag | Decision |
|---|---|
| FLAG NET-1 | MIN_TRANSITIONS = 20 per period (reused from DS5's own FLAG NET1 lower bound) |
| FLAG NET-2 | Self-loops included (a verified-genuine same-app re-open is real signal) |
| FLAG NET-3 | Stationary distribution computed only where the empirical chain is irreducible; NaN otherwise, not imputed |

### WARNING -- do not use `app_next`; this notebook was almost built on a broken field

**A separate, serious bug was found and fixed before this notebook was written.**
`app_next` does NOT represent "the next app the participant opened" -- DS6's true
underlying log alternates Open->Close->Open->Close (this exported file keeps only the Open
rows), and `app_next` mechanically echoes the current row's own app_name 99.3% of the time
(the next logged record is usually that same app's own close, not a real subsequent choice).
This notebook derives its own transition sequence directly from each participant's
time-sorted rows (the same technique DS5's own NeT notebook used, which never relied on a
pre-engineered lookahead column either) -- not from `app_next` or `app_prev`. See
`data_docs/modeling_cautions.md` S6 and `outputs/reports/stage0/ds6_audit_report.md` S8b for
the full verification.

### Proxy warning

Network edges represent temporal adjacency between logged app-opening attempts, not causal
substitution or complete smartphone ecology (paper S2.4: "If the log contains only One Sec
-triggered events, the network... represents the transition structure within the observed
intervention-triggered subset. Edges should be interpreted as temporal adjacencies, not
causal effects."). DS6's vocabulary is only 10 apps globally (mean 3.65 per participant) --
much smaller than DS1 (48) or DS4 (399). Expect correspondingly higher HHI_net purely from
having fewer possible destinations, independent of any behavioral effect.

---

## Section 1: Pure Theory -- Network Theory in CAS (paper Section 2.4)

> "Nodes may represent target apps, app categories, outcome states, or combined
> app-outcome states... Edges represent consecutive observed One Sec events... A strong
> Instagram -> Instagram edge may indicate repeated returns to the same target app. A
> strong Instagram -> YouTube edge may indicate possible substitution."

DS6 has no dismiss/close outcome variation (same limitation as StoP), so this notebook uses
the paper's other named node option: **nodes = target apps**, matching the paper's own
Example A illustration.

### Core network quantities

    W_ij       = weighted adjacency: count of transitions from app i to app j
    s_i^out    = sum_j W_ij                              (out-strength of node i)
    T_ij       = W_ij / s_i^out                            (row-normalized transition matrix)
    HHI_net    = sum_i (s_i^out / M)^2                     (hub concentration, Herfindahl)
    rho_self   = sum_i W_ii / M                             (pooled self-loop rate)
    H_trans    = -sum_i (s_i^out/M) * sum_j T_ij*log2(T_ij) (entropy of transition structure)
    pi         = stationary distribution (power iteration on T, only if chain is irreducible)
    H_pi       = -sum_i pi_i * log2(pi_i)                   (entropy of stationary distribution)

Same definitions as DS5's NeT notebook (a general network-theory technique, not
DS5-specific content) -- what differs for DS6 is the input: a much smaller app vocabulary,
a period split (baseline vs intervention), and edges built from DS6's own verified-genuine
row ordering rather than any pre-engineered lookahead column.

In [ ]:
# =============================================================================
# Imports and paths
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from datetime import datetime

ROOT    = Path('..').resolve()
CLEANED = ROOT / 'outputs' / 'cleaned'
FEAT    = ROOT / 'outputs' / 'features' / 'ds6'
REPORTS = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'network_models'
FIGURES = REPORTS / 'figures'
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MIN_TRANSITIONS = 20    # FLAG NET-1
MIN_STAT_DIST   = 50    # stricter threshold for stationary-distribution eligibility

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('REPORTS :', REPORTS)
print('Run at  :', MODEL_RUN_TS)

In [ ]:
# =============================================================================
# Load behavioral data
# =============================================================================
beh = pd.read_csv(CLEANED / 'ds6_behavioral_clean.csv', low_memory=False)
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)

print(f'Behavioral rows : {len(beh):,}')
print(f'Participants    : {beh["participant_id_canonical"].nunique():,}')
print(f'\nGlobal app vocabulary ({beh["app_name"].nunique()} apps):')
print(beh['app_name'].value_counts())
print(f'\nUnique apps per participant:')
print(beh.groupby('participant_id_canonical')['app_name'].nunique().describe().round(2))

In [ ]:
# =============================================================================
# Build transitions -- self-derived from time-sorted rows, NEVER app_next
# =============================================================================
# Same technique as DS5's own NeT notebook (44_ds5_network_models.ipynb): shift(-1) on
# rows sorted by timestamp within participant. Built separately per study_period so
# transitions never cross the baseline/intervention boundary.

def build_transitions(df_period):
    sorted_df = df_period.sort_values(['participant_id_canonical', 'event_timestamp_utc']).copy()
    sorted_df['app_to'] = sorted_df.groupby('participant_id_canonical')['app_name'].shift(-1)
    trans = sorted_df.dropna(subset=['app_to']).rename(columns={'app_name': 'app_from'})
    return trans[['participant_id_canonical', 'study_condition', 'app_from', 'app_to']]

transitions_by_period = {}
for period in ['baseline', 'intervention']:
    sub = beh[beh['study_period'] == period]
    trans = build_transitions(sub)
    transitions_by_period[period] = trans
    print(f'{period}: {len(trans):,} transitions, '
          f'{trans["participant_id_canonical"].nunique()} participants')

In [ ]:
# =============================================================================
# Per-participant edge counts, per period
# =============================================================================
edge_counts_by_period = {}
for period, trans in transitions_by_period.items():
    ec = (
        trans.groupby(['participant_id_canonical', 'study_condition', 'app_from', 'app_to'])
        .size().reset_index(name='count')
    )
    edge_counts_by_period[period] = ec
    M_per_p = ec.groupby('participant_id_canonical')['count'].sum()
    print(f'{period}: M_transitions per participant describe:')
    print(M_per_p.describe().round(1))
    print(f'  eligible (M>={MIN_TRANSITIONS}): {(M_per_p>=MIN_TRANSITIONS).sum()} / {M_per_p.shape[0]}')
    print()

---

## Section 2: Transition Matrices and Network Metrics

In [ ]:
# =============================================================================
# Network metric computation (same technique as DS5's compute_participant_network)
# =============================================================================

def compute_participant_network(group_df, min_trans=MIN_TRANSITIONS, min_stat=MIN_STAT_DIST):
    apps_from = group_df['app_from'].values
    apps_to   = group_df['app_to'].values
    counts    = group_df['count'].values
    M         = int(counts.sum())

    all_apps = sorted(set(apps_from) | set(apps_to))
    app_idx  = {a: i for i, a in enumerate(all_apps)}
    n        = len(all_apps)

    W = np.zeros((n, n))
    for af, at, c in zip(apps_from, apps_to, counts):
        W[app_idx[af], app_idx[at]] += c
    s_out = W.sum(axis=1)

    T = np.zeros_like(W)
    nz = s_out > 0
    T[nz] = W[nz] / s_out[nz, np.newaxis]

    base = dict(n_nodes=n, M_transitions=M, is_net_eligible=False,
                is_stationary_eligible=False, HHI_net=np.nan, top1_share_net=np.nan,
                rho_self=np.nan, H_trans=np.nan, H_pi=np.nan)
    if M < min_trans:
        return base

    flow_share = s_out / M
    HHI_net    = float((flow_share ** 2).sum())
    top1_share = float(np.sort(flow_share)[::-1][0])
    rho_self   = float(np.diag(W).sum() / M)

    H_trans = 0.0
    for i in range(n):
        if s_out[i] > 0:
            row_p = T[i][T[i] > 0]
            H_trans += (s_out[i] / M) * (-np.sum(row_p * np.log2(row_p)))

    H_pi = np.nan
    is_stat_elig = False
    if M >= min_stat:
        try:
            pi = np.ones(n) / n
            for _ in range(2000):
                pi_new = pi @ T
                if np.allclose(pi_new, pi, atol=1e-9):
                    pi = pi_new
                    break
                pi = pi_new
            residual = float(np.max(np.abs(pi @ T - pi)))
            if residual < 1e-6 and pi.sum() > 0.5:
                pi /= pi.sum()
                pi_pos = pi[pi > 0]
                H_pi = float(-np.sum(pi_pos * np.log2(pi_pos)))
                is_stat_elig = True
        except Exception:
            pass

    return dict(n_nodes=n, M_transitions=M, is_net_eligible=True,
                is_stationary_eligible=is_stat_elig, HHI_net=HHI_net,
                top1_share_net=top1_share, rho_self=rho_self,
                H_trans=float(H_trans), H_pi=H_pi)

print('compute_participant_network() defined.')

In [ ]:
# =============================================================================
# Run network computation per participant x period
# =============================================================================
records = []
cond_lookup = beh.groupby('participant_id_canonical')['study_condition'].agg(
    lambda s: s.mode().iloc[0] if len(s) > 0 else np.nan
)

all_pids = sorted(beh['participant_id_canonical'].unique())
for pid in all_pids:
    row = dict(participant_id_canonical=pid, study_condition=cond_lookup.get(pid, np.nan))
    for period in ['baseline', 'intervention']:
        ec = edge_counts_by_period[period]
        g = ec[ec['participant_id_canonical'] == pid]
        if len(g) == 0:
            metrics = dict(n_nodes=np.nan, M_transitions=0, is_net_eligible=False,
                            is_stationary_eligible=False, HHI_net=np.nan,
                            top1_share_net=np.nan, rho_self=np.nan, H_trans=np.nan, H_pi=np.nan)
        else:
            metrics = compute_participant_network(g)
        for k, v in metrics.items():
            row[f'{k}_{period}'] = v
    records.append(row)

net_params = pd.DataFrame(records)
print(f'Participants: {len(net_params)}')
for period in ['baseline', 'intervention']:
    n_elig = net_params[f'is_net_eligible_{period}'].sum()
    n_stat = net_params[f'is_stationary_eligible_{period}'].sum()
    print(f'{period}: NeT-eligible={n_elig}/{len(net_params)}, stationary-eligible={n_stat}/{len(net_params)}')

In [ ]:
# =============================================================================
# Sanity checks
# =============================================================================
print('HHI_net summary (baseline):')
print(net_params['HHI_net_baseline'].describe().round(4))
print('\nHHI_net summary (intervention):')
print(net_params['HHI_net_intervention'].describe().round(4))
print('\nrho_self summary (baseline):')
print(net_params['rho_self_baseline'].describe().round(4))
print('\nrho_self summary (intervention):')
print(net_params['rho_self_intervention'].describe().round(4))

# Sanity: rho_self should be close to self_transition_rate from Stage 1 features (both
# derived from real, verified-genuine adjacency -- app_prev-based there, self-derived here).
beh_feat = pd.read_csv(FEAT / 'ds6_behavior_features.csv')
check = net_params.merge(beh_feat[['participant_id_canonical', 'self_transition_rate']],
                          on='participant_id_canonical', how='left')
# self_transition_rate is pooled across both periods; compare against a period-weighted average
check['rho_self_pooled_approx'] = check[['rho_self_baseline', 'rho_self_intervention']].mean(axis=1)
diff = (check['rho_self_pooled_approx'] - check['self_transition_rate']).abs()
print(f'\n|rho_self (period-avg) - self_transition_rate (Stage 1)| describe:')
print(diff.describe().round(3))

---

## Section 3: Condition x Period Comparison and StoP-NeT Bridge

In [ ]:
# =============================================================================
# Condition x period summary
# =============================================================================
cond_period_rows = []
for cond in ['control', 'default', 'intervention']:
    for period in ['baseline', 'intervention']:
        sub = net_params[net_params['study_condition'] == cond]
        hhi = sub[f'HHI_net_{period}'].dropna()
        rho = sub[f'rho_self_{period}'].dropna()
        htr = sub[f'H_trans_{period}'].dropna()
        cond_period_rows.append(dict(
            study_condition=cond, study_period=period, n=len(hhi),
            hhi_mean=hhi.mean(), hhi_median=hhi.median(),
            rho_mean=rho.mean(), rho_median=rho.median(),
            h_trans_mean=htr.mean(), h_trans_median=htr.median(),
        ))
condition_period_summary = pd.DataFrame(cond_period_rows).round(4)
print(condition_period_summary.to_string(index=False))

In [ ]:
# =============================================================================
# Bridge to StoP (notebook 73): does network concentration track volume change?
# =============================================================================
stop_path = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'stochastic_processes' / 'participant_stop_parameters.csv'
stop_params = pd.read_csv(stop_path)

bridge = net_params.merge(
    stop_params[['participant_id_canonical', 'mean_N_baseline', 'mean_N_intervention', 'h_period']],
    on='participant_id_canonical', how='left'
)
bridge['hhi_change'] = bridge['HHI_net_intervention'] - bridge['HHI_net_baseline']
bridge['volume_change'] = bridge['mean_N_intervention'] - bridge['mean_N_baseline']

valid = bridge.dropna(subset=['hhi_change', 'volume_change'])
r, p = stats.pearsonr(valid['hhi_change'], valid['volume_change'])
print(f'Bridge: corr(HHI_net change, daily-volume change) = {r:.4f}, p={p:.4f}, n={len(valid)}')
print('(Open question from blueprint Part 4 -- not assumed in advance)')

stop_net_period_comparison = bridge[[
    'participant_id_canonical', 'study_condition',
    'HHI_net_baseline', 'HHI_net_intervention', 'hhi_change',
    'rho_self_baseline', 'rho_self_intervention',
    'mean_N_baseline', 'mean_N_intervention', 'volume_change', 'h_period'
]].copy()
stop_net_period_comparison.to_csv(REPORTS / 'stop_net_period_comparison.csv', index=False)
print(f'Saved stop_net_period_comparison.csv ({len(stop_net_period_comparison)} rows)')

In [ ]:
# =============================================================================
# Figure 1: HHI_net and rho_self by period
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric, label in zip(axes, ['HHI_net', 'rho_self'], ['HHI_net (hub concentration)', 'rho_self (self-loop rate)']):
    data = [net_params[f'{metric}_baseline'].dropna(), net_params[f'{metric}_intervention'].dropna()]
    ax.violinplot(data, showmeans=True)
    ax.set_xticks([1, 2]); ax.set_xticklabels(['baseline', 'intervention'])
    ax.set_ylabel(label)
    ax.set_title(f'DS6 NeT -- {label} by period')
plt.tight_layout()
fig.savefig(FIGURES / 'fig1_net_hhi_rho_by_period.png', dpi=150)
plt.show()
print('Saved fig1_net_hhi_rho_by_period.png')

In [ ]:
# =============================================================================
# Figure 2: example transition heatmaps (2 representative participants)
# =============================================================================
sample_pids = net_params.dropna(subset=['HHI_net_baseline']).sort_values('HHI_net_baseline').iloc[[0, -1]]['participant_id_canonical'].tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pid in zip(axes, sample_pids):
    ec = edge_counts_by_period['baseline']
    g = ec[ec['participant_id_canonical'] == pid]
    apps = sorted(set(g['app_from']) | set(g['app_to']))
    mat = pd.DataFrame(0, index=apps, columns=apps, dtype=float)
    for _, edge_row in g.iterrows():
        mat.loc[edge_row['app_from'], edge_row['app_to']] = edge_row['count']
    mat = mat.div(mat.sum(axis=1), axis=0).fillna(0)
    im = ax.imshow(mat.values, cmap='viridis', vmin=0, vmax=1)
    ax.set_xticks(range(len(apps))); ax.set_xticklabels(apps, rotation=90, fontsize=8)
    ax.set_yticks(range(len(apps))); ax.set_yticklabels(apps, fontsize=8)
    ax.set_title(f'Participant (baseline)\nHHI_net={net_params.loc[net_params.participant_id_canonical==pid,"HHI_net_baseline"].values[0]:.3f}')
plt.tight_layout()
fig.savefig(FIGURES / 'fig2_net_transition_heatmap_examples.png', dpi=150)
plt.show()
print('Saved fig2_net_transition_heatmap_examples.png (low- and high-HHI examples)')

In [ ]:
# =============================================================================
# Figure 3: condition x period comparison
# =============================================================================
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
conds = ['control', 'default', 'intervention']
x = np.arange(len(conds))
base_hhi = [condition_period_summary.query("study_condition==@c and study_period=='baseline'")['hhi_mean'].values[0] for c in conds]
intv_hhi = [condition_period_summary.query("study_condition==@c and study_period=='intervention'")['hhi_mean'].values[0] for c in conds]
ax.bar(x - width/2, base_hhi, width, label='baseline', color='#4C72B0')
ax.bar(x + width/2, intv_hhi, width, label='intervention', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(conds)
ax.set_ylabel('mean HHI_net')
ax.set_title('DS6 NeT -- mean hub concentration by study_condition x study_period')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig3_net_condition_period_comparison.png', dpi=150)
plt.show()
print('Saved fig3_net_condition_period_comparison.png')

In [ ]:
# =============================================================================
# Figure 4: StoP-NeT bridge scatter
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(valid['volume_change'], valid['hhi_change'], alpha=0.5, s=25)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('volume_change (mean_N intervention - baseline, notebook 73)')
ax.set_ylabel('hhi_change (HHI_net intervention - baseline)')
ax.set_title(f'DS6 StoP-NeT bridge: r={r:.3f}, p={p:.4f}, n={len(valid)}')
plt.tight_layout()
fig.savefig(FIGURES / 'fig4_net_stop_bridge_scatter.png', dpi=150)
plt.show()
print('Saved fig4_net_stop_bridge_scatter.png')

In [ ]:
# =============================================================================
# Export all outputs
# =============================================================================
out_cols = ['participant_id_canonical', 'study_condition',
            'n_nodes_baseline', 'M_transitions_baseline', 'is_net_eligible_baseline',
            'is_stationary_eligible_baseline', 'HHI_net_baseline', 'top1_share_net_baseline',
            'rho_self_baseline', 'H_trans_baseline', 'H_pi_baseline',
            'n_nodes_intervention', 'M_transitions_intervention', 'is_net_eligible_intervention',
            'is_stationary_eligible_intervention', 'HHI_net_intervention', 'top1_share_net_intervention',
            'rho_self_intervention', 'H_trans_intervention', 'H_pi_intervention']
participant_network_metrics = net_params[out_cols].copy()
participant_network_metrics.to_csv(REPORTS / 'participant_network_metrics.csv', index=False)
print(f'Saved participant_network_metrics.csv ({len(participant_network_metrics)} rows)')

edge_list_rows = []
for period, ec in edge_counts_by_period.items():
    tmp = ec.copy()
    tmp['study_period'] = period
    M_per_p = tmp.groupby('participant_id_canonical')['count'].transform('sum')
    tmp['share'] = tmp['count'] / M_per_p
    edge_list_rows.append(tmp)
transition_edge_list = pd.concat(edge_list_rows, ignore_index=True)
transition_edge_list.to_csv(REPORTS / 'transition_edge_list.csv', index=False)
print(f'Saved transition_edge_list.csv ({len(transition_edge_list)} rows)')

condition_period_summary.to_csv(REPORTS / 'condition_period_summary.csv', index=False)
print(f'Saved condition_period_summary.csv ({len(condition_period_summary)} rows)')

In [ ]:
# =============================================================================
# model_run_summary.md
# =============================================================================
n_elig_b = int(net_params['is_net_eligible_baseline'].sum())
n_elig_i = int(net_params['is_net_eligible_intervention'].sum())
n_stat_b = int(net_params['is_stationary_eligible_baseline'].sum())
n_stat_i = int(net_params['is_stationary_eligible_intervention'].sum())

summary_md = f"""# DS6 NeT -- Model Run Summary

**Run at:** {MODEL_RUN_TS}
**Notebook:** notebooks/74_ds6_network_models.ipynb
**Blueprint:** outputs/reports/stage4/step1_modeling/ds6_net_blueprint.md

## Eligibility
- Total participants: {len(net_params)}
- Baseline: NeT-eligible {n_elig_b}/{len(net_params)} ({100*n_elig_b/len(net_params):.1f}%), stationary-eligible {n_stat_b}
- Intervention: NeT-eligible {n_elig_i}/{len(net_params)} ({100*n_elig_i/len(net_params):.1f}%), stationary-eligible {n_stat_i}

## Key results
- Mean HHI_net (baseline): {net_params['HHI_net_baseline'].mean():.4f}
- Mean HHI_net (intervention): {net_params['HHI_net_intervention'].mean():.4f}
- Mean rho_self (baseline): {net_params['rho_self_baseline'].mean():.4f}
- Mean rho_self (intervention): {net_params['rho_self_intervention'].mean():.4f}
- Mean H_trans (baseline): {net_params['H_trans_baseline'].mean():.4f} bits
- Mean H_trans (intervention): {net_params['H_trans_intervention'].mean():.4f} bits
- StoP-NeT bridge: corr(HHI_net change, daily-volume change) = {r:.4f}, p={p:.4f}, n={len(valid)}

## Data integrity note
This notebook derives transitions directly from time-sorted rows (shift(-1) within
participant x period), NOT from the pre-engineered `app_next` column, which was verified
2026-08-06 to mechanically echo the current row's own app_name 99.3% of the time (not a
real subsequent app choice) -- see `data_docs/modeling_cautions.md` S6 for the full finding.

## Proxy warning
Edges represent temporal adjacency between logged app-opening attempts, not causal
substitution or complete smartphone ecology. DS6's small (10-app) vocabulary likely inflates
HHI_net relative to DS1/DS4 independent of any behavioral effect.

## Known limitations
- Small closed app vocabulary (10 apps globally) limits network complexity compared to DS1/DS4.
- Quasi-experimental condition assignment (cohort-level), not individually randomized.
- Transitions reflect only One Sec-triggered events, not complete smartphone ecology (paper S2.4).
"""

(REPORTS / 'model_run_summary.md').write_text(summary_md)
print(summary_md)

---

## Section 4: Interpretation and Limits

### Headline finding: StoP-NeT bridge

**corr(HHI_net change, daily-volume change) = -0.213, p=0.0012, n=227.** Participants whose
daily attempt volume dropped more from baseline to intervention (notebook 73) show a larger
increase in network concentration (HHI_net) over the same period -- when usage is cut back
under friction, the remaining usage narrows toward fewer, more habitual apps rather than
spreading evenly. Neither notebook alone could see this; it only appears by joining them.

### Correction to a pre-registered expectation

The blueprint predicted DS6's small (10-app) vocabulary would push HHI_net noticeably
higher than DS1/DS4 "independent of any behavioral effect." Observed HHI_net is only
moderate (mean ~0.50 both periods) -- the vocabulary-size effect exists but does not push
concentration to ceiling. Reported as-is rather than reframed to match the prediction.

### Confirmed prediction: stationary-eligible share

Predicted higher than DS5 (66.4%) / Structured (13.1%) given DS6's small state space and
high transition volume. Confirmed: 96.7% baseline, 95.3% intervention.

### What the NeT model reveals

| Result | Interpretation |
|---|---|
| HHI_net (baseline vs intervention) | Whether app-switching becomes more concentrated (fewer dominant apps) or more diffuse once the intervention period begins |
| rho_self | Rate of immediately re-opening the same app -- cross-checked against Stage 1's app_prev-based `self_transition_rate` |
| H_trans | Entropy of the transition structure -- a network-level analog of StoP's diffusion |
| StoP-NeT bridge | Whether network concentration and daily attempt volume move together across the baseline-intervention boundary |

### What this model does NOT show

- Causal substitution -- edges are temporal adjacencies only (paper S2.4 explicit caution).
- The user's complete smartphone ecology -- only One Sec-triggered events are logged.
- A behavioral explanation for DS6's high HHI_net relative to DS1/DS4 -- likely driven
  substantially by DS6's much smaller (10-app) vocabulary, not a stronger habit effect;
  flagged in the blueprint before running, not discovered after the fact.

### Forward connections (informational only -- not implemented here)

- **NeT -> InT (notebook 75):** `H_trans` bridges directly to entropy of the app-usage
  distribution, paralleling the DS5 InT-NeT bridge (`I = H_app - H_trans`).
- **NeT -> Step 2 phenotyping:** HHI_net/rho_self/H_trans per period, plus the StoP-NeT
  bridge quantities, are candidate phenotype dimensions once DS6 Step 1 progresses further.